<a href="https://colab.research.google.com/github/sap-tarshi-ghosh/AI-assignments/blob/main/langchain_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clear old processes and files
!pkill -9 streamlit
!pkill -9 cloudflared
!rm -f cloudflared-linux-amd64

# 2. Install necessary libraries
!pip install -q streamlit google-generativeai

# 3. Download the stable Cloudflare tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("✅ Step 1 Complete: Environment is ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 31.5 MB/s eta 0:00:00
✅ Step 1 Complete: Environment is ready.


In [3]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai

st.set_page_config(page_title="Final Gemini Bot", page_icon="🤖")
st.title("🤖 My Reliable Gemini Bot")

# --- 1. API KEY INPUT ---
api_key = st.text_input("Paste your Google API Key here:", type="password")

if not api_key:
    st.info("Waiting for your API Key...")
    st.stop()

# --- 2. AUTOMATIC MODEL DISCOVERY ---
genai.configure(api_key=api_key)

if "active_model" not in st.session_state:
    try:
        valid_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        if valid_models:
            st.session_state.active_model = valid_models[0]
            st.success(f"Successfully connected to: {st.session_state.active_model}")
        else:
            st.error("No models found. Ensure Generative AI is enabled in your Google project.")
            st.stop()
    except Exception as e:
        st.error(f"Discovery Error: {e}")
        st.stop()

# --- 3. CHAT LOGIC ---
model = genai.GenerativeModel(st.session_state.active_model)

if "messages" not in st.session_state:
    st.session_state.messages = []

# Display message history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User input
if prompt := st.chat_input("Type your message here..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Get response from the discovered model
    try:
        response = model.generate_content(prompt)
        with st.chat_message("assistant"):
            st.markdown(response.text)
        st.session_state.messages.append({"role": "assistant", "content": response.text})
    except Exception as e:
        st.error(f"Chat failed: {e}")

Overwriting app.py


In [ ]:
# 1. Run the app in the background
!streamlit run app.py &>/content/logs.txt &

# 2. Launch the tunnel (Click the link that appears in the output)
print("Starting tunnel... please wait 10 seconds.")
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

Starting tunnel... please wait 10 seconds.
2026-04-20T07:36:57Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-20T07:36:57Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-20T07:37:02Z INF +--------------------------------------------------------------------------------------------+
2026-04-20T07:37:02Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-20T07:37:02Z INF |  https://ho